In [1]:
"""
Dining Philosophers: exactly 4 philosophers, semaphore based, with an
optional live terminal animation.

Usage
    python3 dining_philosophers.py                    # plain run, log + summary
    python3 dining_philosophers.py --animate          # live ASCII animation
    python3 dining_philosophers.py --animate --no-room  # no deadlock prevention:
                                                       # watch it deadlock
    options: --seconds N   run time (default 5; 20 with --animate)

Deadlock prevention strategy: "Footman" / room semaphore (N-1 admission)
------------------------------------------------------------------------
* One binary semaphore per fork (fork[i]), 4 in total.
* One counting semaphore `room`, initialised to N-1 = 3.
  A philosopher must acquire `room` BEFORE trying to pick up any fork,
  and releases it after putting both forks down.

Why this is deadlock-free
  Deadlock requires a circular wait: all 4 philosophers each holding one
  fork and waiting for the next. `room` guarantees at most 3 philosophers
  are competing for forks at once. With 3 philosophers and 4 forks, the
  pigeonhole principle says at least one of them can always obtain both of
  its forks, eat, and release, so the system always makes progress.

Starvation considerations
  * `room` is a FairSemaphore: waiters are released strictly in arrival
    order, so a hungry philosopher is overtaken only by those already in
    the queue ahead of it (bounded waiting).
  * threading.Semaphore is usually FIFO in CPython but neither Python nor
    POSIX guarantees it, which is why FairSemaphore is used for `room`.
  * Each eating period is finite and a neighbour must re-queue behind a
    waiting philosopher before eating again.
  * The summary reports meals per philosopher so starvation would show.
"""

import argparse
import os
import random
import sys
import threading
import time
from collections import deque

N = 4                      # exactly four philosophers
THINK = (0.3, 1.2)         # think time range (seconds), animation-friendly
EAT = (0.4, 1.0)           # eat time range (seconds)

print_lock = threading.Lock()


def log(msg: str) -> None:
    with print_lock:
        print(msg, flush=True)


class FairSemaphore:
    """Counting semaphore with strict FIFO wake-up (no barging)."""

    def __init__(self, value: int):
        self._value = value
        self._cond = threading.Condition()
        self._queue = deque()          # tickets of waiting threads

    @property
    def value(self) -> int:
        return self._value

    @property
    def waiting(self) -> int:
        return len(self._queue)

    def acquire(self) -> None:
        with self._cond:
            ticket = object()
            self._queue.append(ticket)
            while self._value == 0 or self._queue[0] is not ticket:
                self._cond.wait()
            self._queue.popleft()
            self._value -= 1
            self._cond.notify_all()

    def release(self) -> None:
        with self._cond:
            self._value += 1
            self._cond.notify_all()


# Philosopher states (used by the animation and the safety assertions)
THINKING = "thinking"
QUEUED = "queued"          # waiting for the room semaphore
NEEDS_FORK = "needs fork"  # in the room (or room disabled), waiting for left fork
ONE_FORK = "1 fork"        # holds left fork, waiting for right fork
EATING = "eating"


class DiningTable:
    def __init__(self, use_room: bool = True, hold_delay: float = 0.0, quiet: bool = False):
        self.n = N
        self.use_room = use_room
        self.hold_delay = hold_delay     # pause between the two fork pick-ups
        self.quiet = quiet               # suppress per-meal log lines (animation)
        self.fork = [threading.Semaphore(1) for _ in range(N)]
        self.room = FairSemaphore(N - 1)                  # deadlock prevention
        self.fork_owner = [None] * N                      # for display only
        self.meals = [0] * N
        self.state = [THINKING] * N
        self.stop = threading.Event()

    # Philosopher i uses fork i (left) and fork (i+1) % n (right)
    def pick_up(self, i: int) -> None:
        left, right = i, (i + 1) % self.n
        if self.use_room:
            self.state[i] = QUEUED
            self.room.acquire()                           # at most 3 enter
        self.state[i] = NEEDS_FORK
        self.fork[left].acquire()
        self.fork_owner[left] = i
        self.state[i] = ONE_FORK
        if self.hold_delay:
            time.sleep(self.hold_delay)
        self.fork[right].acquire()
        self.fork_owner[right] = i

    def put_down(self, i: int) -> None:
        left, right = i, (i + 1) % self.n
        # Update the shared bookkeeping BEFORE releasing anything. If the forks
        # were released first, a neighbour could start eating while this
        # philosopher still shows EATING and trip the safety assertion.
        self.state[i] = THINKING
        self.fork_owner[left] = None
        self.fork_owner[right] = None
        self.fork[left].release()
        self.fork[right].release()
        if self.use_room:
            self.room.release()

    def philosopher(self, i: int) -> None:
        rng = random.Random(i)
        while not self.stop.is_set():
            # Without the room, short thinking makes simultaneous hunger (and
            # therefore the circular wait) likely enough to demonstrate.
            time.sleep(rng.uniform(*(THINK if self.use_room else (0.0, 0.2))))
            self.pick_up(i)
            self.state[i] = EATING
            # Safety check: neighbours must never eat at the same time.
            assert self.state[(i + 1) % self.n] != EATING
            assert self.state[(i - 1) % self.n] != EATING
            self.meals[i] += 1
            if not self.quiet:
                log(f"Philosopher {i} eating (meal #{self.meals[i]})")
            time.sleep(rng.uniform(*EAT))                 # eat
            self.put_down(i)

    def deadlocked(self) -> bool:
        return all(s == ONE_FORK for s in self.state)


# ---------------------------------------------------------------- animation

RESET = "\033[0m"
COLOR = {THINKING: "\033[90m", QUEUED: "\033[33m", NEEDS_FORK: "\033[33m",
         ONE_FORK: "\033[38;5;208m", EATING: "\033[32m"}
RED = "\033[31;1m"


HTML_COLOR = {THINKING: "#888888", QUEUED: "#c99a00", NEEDS_FORK: "#c99a00",
              ONE_FORK: "#e0761f", EATING: "#1f9d68"}


def frame(table: DiningTable, elapsed: float, dead: bool, html: bool = False) -> str:
    def p(i: int) -> str:
        s = table.state[i]
        label = f"P{i} {s:<10}"
        if html:
            return f'<span style="color:{HTML_COLOR[s]};font-weight:bold">{label}</span>'
        return f"{COLOR[s]}{label}{RESET}"

    def f(i: int) -> str:
        o = table.fork_owner[i]
        return f"F{i}[{'--' if o is None else 'P' + str(o)}]"

    if table.use_room:
        room = f"room {table.room.value}/{N - 1} queue {table.room.waiting}"
    else:
        room = "room OFF"

    lines = [
        f"Dining Philosophers (4, semaphores)   t = {elapsed:5.1f}s",
        "",
        f"            {p(0)}",
        f"     {f(0)}                 {f(1)}",
        f"{p(3)}   {room:^22}   {p(1)}",
        f"     {f(3)}                 {f(2)}",
        f"            {p(2)}",
        "",
        "meals: " + "  ".join(f"P{i}={m}" for i, m in enumerate(table.meals)),
    ]
    if dead:
        msg = "DEADLOCK: every philosopher holds one fork and waits for the next."
        lines += ["", f'<span style="color:#d23b36;font-weight:bold">{msg}</span>' if html
                  else f"{RED}{msg}{RESET}"]
    return "\n".join(lines)


def run(seconds: float, animate: bool, use_room: bool) -> None:
    table = DiningTable(use_room=use_room,
                        hold_delay=0.0 if use_room else 0.4,
                        quiet=animate)
    threads = [threading.Thread(target=table.philosopher, args=(i,), daemon=True)
               for i in range(N)]
    start = time.time()
    for t in threads:
        t.start()

    dead = False
    if animate:
        notebook = in_notebook()
        handle = None
        if notebook:
            from IPython.display import HTML, display
            # One output cell that is updated in place on every frame.
            handle = display(HTML(""), display_id=True)
        else:
            os.system("")                                 # enables ANSI colours on Windows terminals
            sys.stdout.write("\033[2J\033[?25l")          # clear screen, hide cursor
        try:
            while time.time() - start < seconds:
                dead = table.deadlocked()
                if notebook:
                    text = frame(table, time.time() - start, dead, html=True)
                    handle.update(HTML('<pre style="font-family:monospace;line-height:1.35;'
                                       'font-size:14px">' + text + "</pre>"))
                else:
                    text = frame(table, time.time() - start, dead)
                    sys.stdout.write("\033[H" + text + "\033[J")
                    sys.stdout.flush()
                if dead:
                    time.sleep(1.5)                       # let the user see it
                    break
                time.sleep(0.1)
        except KeyboardInterrupt:
            pass
        finally:
            if not notebook:
                sys.stdout.write("\033[?25h\n")           # restore cursor
    else:
        time.sleep(seconds)
        dead = table.deadlocked()

    table.stop.set()
    if not dead:
        for t in threads:
            t.join(timeout=2.0)

    print("\n=== Summary ===")
    for i, m in enumerate(table.meals):
        print(f"Philosopher {i}: {m} meals")
    print(f"Max/min meals ratio: {max(table.meals) / max(1, min(table.meals)):.2f}")
    print("Deadlock detected!" if dead else "No deadlock: all philosophers finished cleanly.")


def run_gui(seconds: float, use_room: bool) -> None:
    """Graphical animation in a tkinter window (standard library, no installs)."""
    import math
    import tkinter as tk

    table = DiningTable(use_room=use_room,
                        hold_delay=0.0 if use_room else 0.4,
                        quiet=True)
    threads = [threading.Thread(target=table.philosopher, args=(i,), daemon=True)
               for i in range(N)]

    W = 560
    cx = cy = W / 2
    colors = {THINKING: "#8ea2b5", QUEUED: "#d9a21b", NEEDS_FORK: "#d9a21b",
              ONE_FORK: "#e0761f", EATING: "#23996a"}

    root = tk.Tk()
    root.title("Dining Philosophers (4, semaphores)")
    cv = tk.Canvas(root, width=W, height=W + 40, bg="#eceff3", highlightthickness=0)
    cv.pack()

    def pol(r, deg):
        a = math.radians(deg)
        return cx + r * math.cos(a), cy + r * math.sin(a)

    def ang(i):                      # philosopher i: top, right, bottom, left
        return -90 + 90 * i

    start = time.time()
    state = {"dead": False, "closed": False}
    for t in threads:
        t.start()

    def draw():
        if state["closed"]:
            return
        elapsed = time.time() - start
        dead = table.deadlocked()
        cv.delete("all")
        cv.create_oval(cx - 160, cy - 160, cx + 160, cy + 160, fill="#dde4ea", outline="")
        for i in range(N):
            x, y = pol(102, ang(i))
            eating = table.state[i] == EATING
            cv.create_oval(x - 25, y - 25, x + 25, y + 25,
                           fill="#9ad8bd" if eating else "#f7f9fb", outline="#cbd4dc", width=2)
        # forks: free forks rest between philosophers, held forks sit by the owner's plate
        for f in range(N):
            owner = table.fork_owner[f]
            if owner is None:
                x, y = pol(122, ang(f) - 45)
            else:
                x, y = pol(100, ang(owner) + (-20 if f == owner else 20))
            a = math.radians(math.degrees(math.atan2(y - cy, x - cx)))
            dx, dy = math.cos(a) * 17, math.sin(a) * 17
            cv.create_line(x - dx, y - dy, x + dx, y + dy, width=5, fill="#3b4a58",
                           capstyle="round")
            cv.create_text(x + dy * 1.6, y - dx * 1.6, text=f"F{f}", fill="#586a7b",
                           font=("Consolas", 9))
        for i in range(N):
            x, y = pol(208, ang(i))
            s = "deadlocked" if dead else table.state[i]
            cv.create_oval(x - 42, y - 42, x + 42, y + 42,
                           fill="#d23b36" if dead else colors[table.state[i]],
                           outline="white", width=3)
            cv.create_text(x, y - 8, text=f"P{i}", fill="#0e1a24", font=("Segoe UI", 15, "bold"))
            cv.create_text(x, y + 12, text=s, fill="#0e1a24", font=("Consolas", 9))
            cv.create_text(x, y + 56, text=f"meals {table.meals[i]}", fill="#586a7b",
                           font=("Consolas", 9))
        if use_room:
            cv.create_text(cx, cy - 8, text=f"room {table.room.value}/{N - 1}",
                           fill="#2457c5", font=("Consolas", 14, "bold"))
            cv.create_text(cx, cy + 14, text=f"queue {table.room.waiting}",
                           fill="#586a7b", font=("Consolas", 10))
        else:
            cv.create_text(cx, cy, text="room OFF", fill="#d23b36", font=("Consolas", 14, "bold"))
        footer = f"t = {elapsed:5.1f}s    "
        footer += "DEADLOCK: everyone holds one fork and waits" if dead else "running"
        cv.create_text(cx, W + 20, text=footer, font=("Consolas", 11),
                       fill="#d23b36" if dead else "#14202b")
        state["dead"] = dead
        if dead or elapsed >= seconds:
            root.after(1500 if dead else 0, close)
            return
        root.after(50, draw)

    def close():
        if not state["closed"]:
            state["closed"] = True
            root.destroy()

    root.protocol("WM_DELETE_WINDOW", close)
    root.after(0, draw)
    root.mainloop()

    table.stop.set()
    if not state["dead"]:
        for t in threads:
            t.join(timeout=2.0)
    print("\n=== Summary ===")
    for i, m in enumerate(table.meals):
        print(f"Philosopher {i}: {m} meals")
    print(f"Max/min meals ratio: {max(table.meals) / max(1, min(table.meals)):.2f}")
    print("Deadlock detected!" if state["dead"] else "No deadlock: all philosophers finished cleanly.")


def in_notebook() -> bool:
    return "ipykernel" in sys.modules


def main() -> None:
    ap = argparse.ArgumentParser(description="Dining philosophers (4) with semaphores")
    ap.add_argument("--animate", action="store_true", help="live terminal animation")
    ap.add_argument("--gui", action="store_true", help="graphical window animation (tkinter)")
    ap.add_argument("--no-room", action="store_true",
                    help="disable the room semaphore (deadlock is then possible)")
    ap.add_argument("--seconds", type=float, default=None, help="how long to run")
    # parse_known_args ignores extra arguments such as Jupyter's "--f=...kernel.json"
    a, _ = ap.parse_known_args()
    # Default to the graphical window when run without options (most reliable in
    # notebooks and IDEs); fall back to text if tkinter is unavailable.
    gui = a.gui or not (a.animate or a.seconds is not None or a.no_room)
    if gui:
        try:
            import tkinter  # noqa: F401
        except ImportError:
            print("tkinter not available, using text animation instead.")
            gui = False
    if gui:
        run_gui(a.seconds if a.seconds is not None else 20.0, not a.no_room)
        return
    animate = a.animate or in_notebook()
    seconds = a.seconds if a.seconds is not None else (20.0 if animate else 5.0)
    run(seconds, animate, not a.no_room)


if __name__ == "__main__":
    main()


=== Summary ===
Philosopher 0: 10 meals
Philosopher 1: 11 meals
Philosopher 2: 10 meals
Philosopher 3: 11 meals
Max/min meals ratio: 1.10
No deadlock: all philosophers finished cleanly.
